In [1]:
# LIBRARIES FOR FEATURE ENGINEERING

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, recall_score, precision_score, f1_score 
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score 

In [2]:
cleaned_customer_transaction_data=pd.read_csv(r"C:/Users/user/Downloads/Fraudulent_Transaction_Detection_for_Finlora_Company/Finlora_dataset/artifacts/EDA_Data.csv")

In [3]:
cleaned_customer_transaction_data.head()

,Unnamed: 0,transaction_id,customer_id,timestamp,home_country,source_currency,dest_currency,channel,amount_src,amount_usd,...,corridor_risk,is_fraud,hour,day_of_week,is weekend,month,account_age_bucket,device_trust_score_bucket,ip_risk_score_bucket,amount_usd_bucket
0,0,fee8542d-8ee6-4b0d-9671-c294dd08ed26,402cccc9-28de-45b3-9af7-cc5302aa1f93,2022-10-03 18:40:59.468549+00:00,US,USD,CAD,atm,278.19,278.19,...,0.0,0,18,0,0,10,180-365 days,0.5-0.7,<0.2,$250-500
1,1,bfdb9fc1-27fe-4a85-b043-4d813d679259,67c2c6b3-ef0a-4777-a3f1-c84a851bb6ad,2022-10-03 20:39:38.468549+00:00,CA,CAD,MXN,web,208.51,154.29,...,0.0,0,20,0,0,10,>365,0.3-0.5,0.4-0.6,<$250
2,2,fc855034-3ea5-4993-9afa-b511d93fe5e8,6d0d9b27-fa26-45f8-93b1-2df29d182d9c,2022-10-03 23:02:43.468549+00:00,US,USD,CNY,mobile,160.33,160.33,...,0.0,0,23,0,0,10,>365,>0.9,0.4-0.6,<$250
3,3,2cf8c08e-42ec-444d-a755-34b9a2a0a4ca,7bd5200c-5d19-44f0-9afe-8b339a05366b,2022-10-04 01:08:53.468549+00:00,US,USD,EUR,mobile,59.41,59.41,...,0.0,0,1,1,0,10,90-180 days,0.5-0.7,0.4-0.6,<$250
4,4,d907a74d-b426-438d-97eb-dbe911aca91c,70a93d26-8e3a-4179-900c-a4a7a74d08e5,2022-10-04 09:35:03.468549+00:00,US,USD,INR,mobile,200.96,200.96,...,0.0,0,9,1,0,10,180-365 days,0.7-0.9,<0.2,<$250


Threshold-based features were created based on key behavioral and risk indicators identified during the exploratory analysis. These include low KYC verification levels, location mismatches between the registered country and IP address and high risk transaction characteristics such as high transaction amounts, elevated IP risk scores, and low device trust scores. These features were identified as high_risk thresholds due to their significantly elevated fraud rates. These thresholds were converted into binary risk indicators to enhance the model's ability to detect suspicious transactions.

In [7]:
# creating a threshold based features from the following risk_signal
cleaned_customer_transaction_data['timestamp'] = pd.to_datetime(cleaned_customer_transaction_data['timestamp'])
cleaned_customer_transaction_data['late_night_hour']=((cleaned_customer_transaction_data['hour']>= 3) & (cleaned_customer_transaction_data['hour'] <= 7)).astype(int)
cleaned_customer_transaction_data['amount_high']=(cleaned_customer_transaction_data['amount_usd']> 1000).astype(int)
cleaned_customer_transaction_data['high_ip_risk'] =(cleaned_customer_transaction_data['ip_risk_score'] > 0.8).astype(int)
cleaned_customer_transaction_data['low_device_trust'] =(cleaned_customer_transaction_data['device_trust_score']<0.5).astype(int)
cleaned_customer_transaction_data['new_account']= (cleaned_customer_transaction_data['account_age_days']>=30)& (cleaned_customer_transaction_data['account_age_days']<=90).astype(int)
cleaned_customer_transaction_data['very_new_account']=(cleaned_customer_transaction_data['account_age_days']<30).astype(int)
cleaned_customer_transaction_data['velocity_spike']=(cleaned_customer_transaction_data['txn_velocity_1h']>=3).astype(int)

high_risk_signal_features = cleaned_customer_transaction_data[['late_night_hour', 'amount_high', 'high_ip_risk', 'low_device_trust', 'new_account', 'velocity_spike']]
high_risk_signal_features.head()


,late_night_hour,amount_high,high_ip_risk,low_device_trust,new_account,velocity_spike
0,0,0,0,0,False,0
1,0,0,0,1,False,0
2,0,0,0,0,False,0
3,0,0,0,0,False,0
4,0,0,0,0,False,0


In [10]:
list(cleaned_customer_transaction_data.columns)

['Unnamed: 0',
 'timestamp',
 'home_country',
 'source_currency',
 'dest_currency',
 'channel',
 'amount_src',
 'amount_usd',
 'fee',
 'exchange_rate_src_to_dest',
 'new_device',
 'ip_country',
 'location_mismatch',
 'ip_risk_score',
 'kyc_tier',
 'account_age_days',
 'device_trust_score',
 'chargeback_history_count',
 'risk_score_internal',
 'txn_velocity_1h',
 'txn_velocity_24h',
 'corridor_risk',
 'is_fraud',
 'hour',
 'day_of_week',
 'is weekend',
 'month',
 'late_night_hour',
 'amount_high',
 'high_ip_risk',
 'low_device_trust',
 'new_account',
 'very_new_account',
 'velocity_spike']

### Feature Selection

In [64]:
#Dropping all temporary buckets column
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['account_age_bucket','device_trust_score_bucket','ip_risk_score_bucket','amount_usd_bucket'], axis=1,  errors='ignore' )
#Also i will be dropping all identifiers column (Ids)
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['transaction_id','customer_id','device_id','ip_address'], axis=1,  errors='ignore')
#Dropping some irrelevant variables
cleaned_customer_transaction_data = cleaned_customer_transaction_data.drop(['chargeback_history_count','month','exchange_rate_src_to_dest','Unnamed: 0'], axis = 1,  errors='ignore')

### Defining Categorical feautres 

In [65]:
categorical_features = cleaned_customer_transaction_data.select_dtypes(include=['object','bool']).columns
categorical_features

Index(['home_country', 'source_currency', 'dest_currency', 'channel',
       'new_device', 'ip_country', 'location_mismatch', 'kyc_tier'],
      dtype='object')

In [72]:
#Definingbnumerical features
numerical_features = cleaned_customer_transaction_data.select_dtypes(include=['int','float']).columns.drop('is_fraud')
numerical_features 

Index(['amount_src', 'amount_usd', 'fee', 'ip_risk_score', 'account_age_days',
       'device_trust_score', 'risk_score_internal', 'txn_velocity_1h',
       'txn_velocity_24h', 'corridor_risk', 'hour', 'day_of_week',
       'is weekend', 'late_night_hour', 'amount_high', 'high_ip_risk',
       'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [78]:
print(f"Categorical:{len(categorical_features)}")
print(f"Numerical:{len(numerical_features)}")
print(f"Dataset: {cleaned_customer_transaction_data.shape}")

Categorical:8
Numerical:20
Dataset: (10804, 30)


In [83]:
cleaned_customer_transaction_data.columns 

Index(['timestamp', 'home_country', 'source_currency', 'dest_currency',
       'channel', 'amount_src', 'amount_usd', 'fee', 'new_device',
       'ip_country', 'location_mismatch', 'ip_risk_score', 'kyc_tier',
       'account_age_days', 'device_trust_score', 'risk_score_internal',
       'txn_velocity_1h', 'txn_velocity_24h', 'corridor_risk', 'is_fraud',
       'hour', 'day_of_week', 'is weekend', 'late_night_hour', 'amount_high',
       'high_ip_risk', 'low_device_trust', 'new_account', 'very_new_account',
       'velocity_spike'],
      dtype='object')

In [88]:
cleaned_customer_transaction_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10804 entries, 0 to 10803
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   timestamp            10804 non-null  datetime64[ns, UTC]
 1   home_country         10779 non-null  object             
 2   source_currency      10804 non-null  object             
 3   dest_currency        10804 non-null  object             
 4   channel              10804 non-null  object             
 5   amount_src           10804 non-null  float64            
 6   amount_usd           10804 non-null  float64            
 7   fee                  10804 non-null  float64            
 8   new_device           10804 non-null  bool               
 9   ip_country           10774 non-null  object             
 10  location_mismatch    10804 non-null  bool               
 11  ip_risk_score        10804 non-null  float64            
 12  kyc_tier          

In [98]:
cleaned_customer_transaction_data.to_csv("../Finlora_Dataset/artifacts/Engineered_Data.csv", index=False)